# EPO Data Visualisations & Matching Validation

This notebook loads the processed pkl files generated by `data_processing.py` and produces:
1. Descriptive statistics and visualisations of the EPO dataset
2. Matching validation results (confusion matrices) from manually annotated samples

## Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Paths
DATA_DIR = os.path.join(os.getcwd(), '../Data')
MATCHING_DIR = os.path.join(DATA_DIR, 'Matching')

## Load Processed Data

These pkl files are generated by `data_processing.py` (run via SLURM).

In [ ]:
# Full cleaned dataset (all case types)
t_df_clean = pd.read_pickle(os.path.join(DATA_DIR, 'EPO_Data.pkl'))
print(f'EPO_Data.pkl: {len(t_df_clean):,} rows')

# Patent Refusal subset with Outcome
binary_refusal_df = pd.read_pickle(os.path.join(DATA_DIR, 'Train&TestData_1.0_PatentRefusal.pkl'))
print(f'PatentRefusal: {len(binary_refusal_df):,} rows')

# Opposition Division with Outcome
opposition_df_clean = pd.read_pickle(os.path.join(DATA_DIR, 'Train&TestData_2.0_OppositionDivision'))
print(f'OppositionDivision: {len(opposition_df_clean):,} rows')

## 1. Descriptive Statistics

In [ ]:
df = t_df_clean

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 8))

# Court Type by Procedure Language
sns.countplot(
    data=df,
    x='Court Type', hue='Procedure Language',
    palette='dark', alpha=.6, ax=ax1
)

# Cases over time
date_group = df.groupby(['Year'], as_index=False).count()
sns.lineplot(data=date_group, x='Year', y='Reference', ax=ax2)

# Board codes
sns.countplot(
    data=df.sort_values('Board Code', ascending=True),
    y='Board Code', hue='Board Code',
    palette='dark', dodge=False, alpha=.8, ax=ax3
)
if ax3.get_legend() is not None:
    ax3.get_legend().remove()

plt.tight_layout()
plt.show()

### Matched Articles Distribution

In [ ]:
# Frequency of individual matched articles
matched_articles_counts = df['Matched Articles'].explode().value_counts()

plt.figure(figsize=(10, 6))
matched_articles_counts.plot(kind='bar', width=0.75)
plt.title('Frequency of Matched Articles')
plt.xlabel('Matched Articles')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Frequency of article combinations
matched_articles_combinations = df['Matched Articles'].apply(lambda x: tuple(sorted(x))).value_counts()
matched_articles_combinations = matched_articles_combinations.sort_values(ascending=True)

plt.figure(figsize=(12, 8))
matched_articles_combinations.plot(kind='barh', width=0.75)
plt.title('Frequency of Matched Articles Combinations')
plt.xlabel('Frequency')
plt.ylabel('Matched Articles Combinations')
plt.show()

### Case Type Distribution

In [ ]:
sns.countplot(data=t_df_clean, x='Matches')
plt.title('Case Type Distribution')
plt.show()

### Opposition Division Sub-classification Distribution

In [ ]:
sns.countplot(data=opposition_df_clean, x='Matches')
plt.title('Opposition Division: Appellant Type')
plt.xlabel('Appellant Type (1=Patentee, 2=Opponent, 3=Both)')
plt.show()

### Patent Refusal Outcome Distribution

In [ ]:
sns.countplot(data=binary_refusal_df, x='Outcome')
plt.title('Patent Refusal: Outcome Distribution')
plt.show()

print(f'Total PR cases with known outcome: {len(binary_refusal_df):,}')
print(binary_refusal_df['Outcome'].value_counts())

### Opposition Division Outcome Distribution

In [ ]:
# Full OD (including Unknown)
sns.countplot(data=opposition_df_clean, x='Outcome')
plt.title('Opposition Division: Outcome Distribution')
plt.show()

opposition_binary = opposition_df_clean.loc[opposition_df_clean['Outcome'] != 'Unknown']
print(f'Total OD cases: {len(opposition_df_clean):,}')
print(f'OD cases with known outcome: {len(opposition_binary):,}')
print(opposition_df_clean['Outcome'].value_counts())

### Combined Outcome Analysis

Outcome distribution across Patent Refusal and Opposition Division combined.

In [ ]:
# Merge PR and OD for combined analysis
merged_df = pd.concat([opposition_df_clean, binary_refusal_df], ignore_index=True)

# Categorise: Patent Refusal vs Opposition Division
merged_df['Category'] = merged_df['Matches'].apply(
    lambda x: 'Patent Refusal' if x == 'Patent Refusal' else 'Opposition Division'
)

# Pivot: outcome counts per category
pivot_table = merged_df.pivot_table(index='Category', columns='Outcome', aggfunc='size', fill_value=0)
pivot_table_percentage = pivot_table.div(pivot_table.sum(axis=1), axis=0) * 100

ax = pivot_table_percentage.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title('Outcome Distribution: Patent Refusal vs Opposition Division')
plt.xlabel('Category')
plt.ylabel('Percentage')
plt.legend(title='Outcome')

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', label_type='center')

plt.show()

In [ ]:
# Outcome distribution by Matched Articles
exploded_df = merged_df.explode('Matched Articles')

pivot_table = exploded_df.pivot_table(index='Matched Articles', columns='Outcome', aggfunc='size', fill_value=0)
pivot_table_percentage = pivot_table.div(pivot_table.sum(axis=1), axis=0) * 100

ax = pivot_table_percentage.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title('Outcome Distribution by Matched Articles')
plt.xlabel('Matched Articles')
plt.ylabel('Percentage')
plt.legend(title='Outcome')

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', label_type='center')

plt.show()

### Inventive Step Outcome by Year

In [ ]:
# Inventive Step cases from the PR subset
inventive_step_mask = binary_refusal_df['Matched Articles'].astype(str).str.contains('Inventive Step', case=False, na=False)
inventive_step_cases = binary_refusal_df[inventive_step_mask]
inventive_step_2015_2024 = inventive_step_cases[inventive_step_cases['Year'] >= 2015]

outcome_by_year = pd.crosstab(inventive_step_2015_2024['Year'], inventive_step_2015_2024['Outcome'])

outcome_by_year.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title('Inventive Step Outcome by Year (2015-2024)')
plt.ylabel('Count')
plt.show()

print('='*60)
print('INVENTIVE STEP CASES OUTCOME SUMMARY (2015-2024)')
print('='*60)
print('\nOutcome counts by year:')
print(outcome_by_year)

print('\nOutcome percentages by year:')
outcome_percentages = outcome_by_year.div(outcome_by_year.sum(axis=1), axis=0) * 100
print(outcome_percentages.round(2))

---

## 2. Matching Validation

This section evaluates the accuracy of the spaCy Matcher patterns used in `data_processing.py` to classify cases and extract outcomes.

For each matcher, a random sample of 50 cases was drawn, manually annotated with the true label in a `REAL` column, and the matcher's prediction compared against it. The annotated `.xlsx` files are stored in `../Data/Matching/`.

| File | What it validates | Labels |
|---|---|---|
| `CaseTypeTest_50.xlsx` | Case type classification (Step 4) | Opposition Division, Patent Refusal, Other |
| `OP_Test_50.xlsx` | Opposition Division appellant sub-type (Step 6) | 1 (Patentee), 2 (Opponent), 3 (Both), Other |
| `PF_Outcome_Test_50.xlsx` | Patent Refusal outcome extraction (Step 7) | Affirmed, Reversed, Unknown |
| `OP_Outcome_Test_50.xlsx` | Opposition Division outcome extraction (Step 7) | Affirmed, Reversed, Unknown |

### 2.1 Case Type Classification

Validates the matcher that classifies cases as **Opposition Division**, **Patent Refusal**, or **Other** based on Summary Facts text.

In [ ]:
case_type_test = pd.read_excel(os.path.join(MATCHING_DIR, 'CaseTypeTest_50.xlsx'))

disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(case_type_test['REAL'], case_type_test['Matches']),
    display_labels=['Opposition Division', 'Other', 'Patent Refusal']
)
disp.plot(cmap='Blues')
plt.title('Case Type Classification (n=50)')
plt.show()

# Accuracy
correct = (case_type_test['REAL'] == case_type_test['Matches']).sum()
print(f'Accuracy: {correct}/{len(case_type_test)} = {correct/len(case_type_test)*100:.1f}%')

### 2.2 Opposition Division Sub-classification

Validates the matcher that identifies whether the appeal was filed by the **Patentee (1)**, **Opponent (2)**, **Both (3)**, or **Other**.

In [ ]:
op_type_test = pd.read_excel(os.path.join(MATCHING_DIR, 'OP_Test_50.xlsx'))
op_type_test['REAL'] = op_type_test['REAL'].astype(str)
op_type_test['Matches'] = op_type_test['Matches'].astype(str)

disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(op_type_test['REAL'], op_type_test['Matches']),
    display_labels=['Patentee', 'Opponent', 'Both', 'Other']
)
disp.plot(cmap='Blues')
plt.title('Opposition Division Sub-classification (n=50)')
plt.show()

correct = (op_type_test['REAL'] == op_type_test['Matches']).sum()
print(f'Accuracy: {correct}/{len(op_type_test)} = {correct/len(op_type_test)*100:.1f}%')

### 2.3 Patent Refusal Outcome Extraction

Validates the outcome matcher (Affirmed / Reversed / Unknown) on Patent Refusal Order text.

In [ ]:
pf_outcome_test = pd.read_excel(os.path.join(MATCHING_DIR, 'PF_Outcome_Test_50.xlsx'))

disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(pf_outcome_test['REAL'], pf_outcome_test['Matches']),
    display_labels=['Affirmed', 'Reversed', 'Unknown']
)
disp.plot(cmap='Blues')
plt.title('Patent Refusal Outcome Extraction (n=50)')
plt.show()

correct = (pf_outcome_test['REAL'] == pf_outcome_test['Matches']).sum()
print(f'Accuracy: {correct}/{len(pf_outcome_test)} = {correct/len(pf_outcome_test)*100:.1f}%')

### 2.4 Opposition Division Outcome Extraction

Validates the outcome matcher (Affirmed / Reversed / Unknown) on Opposition Division Order text.

In [ ]:
op_outcome_test = pd.read_excel(os.path.join(MATCHING_DIR, 'OP_Outcome_Test_50.xlsx'))

disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(op_outcome_test['REAL'], op_outcome_test['MatchesOutcome']),
    display_labels=['Affirmed', 'Reversed', 'Unknown']
)
disp.plot(cmap='Blues')
plt.title('Opposition Division Outcome Extraction (n=50)')
plt.show()

correct = (op_outcome_test['REAL'] == op_outcome_test['MatchesOutcome']).sum()
print(f'Accuracy: {correct}/{len(op_outcome_test)} = {correct/len(op_outcome_test)*100:.1f}%')

### 2.5 Matching Validation Summary

In [ ]:
# Summary table
results = []

for name, test_df, pred_col in [
    ('Case Type', case_type_test, 'Matches'),
    ('OD Sub-classification', op_type_test, 'Matches'),
    ('PR Outcome', pf_outcome_test, 'Matches'),
    ('OD Outcome', op_outcome_test, 'MatchesOutcome'),
]:
    n = len(test_df)
    correct = (test_df['REAL'] == test_df[pred_col]).sum()
    acc = correct / n * 100
    results.append({'Matcher': name, 'n': n, 'Correct': correct, 'Accuracy (%)': f'{acc:.1f}'})

summary = pd.DataFrame(results)
summary